**Automated script**

In [0]:
%pip install xgboost

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
# ========================================================
# Imports
# ========================================================
import pyspark.sql.functions as F
from pyspark.sql import Window
import joblib
import xgboost as xgb
import pandas as pd
from delta.tables import DeltaTable
from sklearn.base import BaseEstimator, TransformerMixin

# ========================================================
# 0. Custom Transformer (Required by Saved Model)
# ========================================================
class ToNumeric(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for c in self.columns:
            if c in X.columns:
                X[c] = pd.to_numeric(X[c], errors="coerce")
        return X

# ========================================================
# 1. Load Regression Model + Preprocess
# ========================================================
artifact_dir = "/Workspace/Users/suhani.thakur@hp.com/Shared/xgb_checkpoints"

xgb_loaded = xgb.XGBRegressor()
xgb_loaded.load_model(f"{artifact_dir}/best_model1.json")
preprocess_loaded = joblib.load(f"{artifact_dir}/preprocess_tuned.pkl")
_cat = list(preprocess_loaded.transformers[0][2])
_num = list(preprocess_loaded.transformers[1][2])

# ========================================================
# 2. Load SNI Label Model
# ========================================================
sni_label_model_path = "/Workspace/Users/suhani.thakur@hp.com/Shared/artifacts/model.joblib"
sni_label_model = joblib.load(sni_label_model_path)

_label_preproc = sni_label_model.named_steps["preprocess"]
_label_feature_cols = []
for _, _, cols in _label_preproc.transformers:
    _label_feature_cols.extend(cols)

# ========================================================
# 3. Build Required Column List BEFORE Loading Data
#    (OPT-2: Column Pruning — know what you need upfront)
# ========================================================
source_table = (
    "supplychain.shipped_not_invoiced."
    "fact_enterprise_shipped_not_invoiced_legacy_combined_reporting_view"
)
target_table = (
    "innovation_supplychain_stg.sni_otc."
    "sni_prediction_regression_classification_v2"
)

id_cols = [
    "Sales_Order_Identifier",
    "Sales_Order_Line_Item_Identifier",
    "Shipment_Identifier",
    "Shipment_Line_Item_Identifier",
    "Shipment_Date"
]

feature_cols = _cat + _num

_date_cols_for_engineering = [
    "Sales_Order_Header_Create_Date",
    "Proof_Of_Delivery_Date",
    "Planned_Shipment_Date",
    "planned_invoice_date",
    "Crdd",
    "Tdd"
]

# Combine all needed columns (+ Dm_Date for dedup, + Dm_Aging_Days as target)
all_needed_cols = list(dict.fromkeys(
    id_cols
    + feature_cols
    + _label_feature_cols
    + _date_cols_for_engineering
    + ["Dm_Aging_Days", "Dm_Date"]
))

# ========================================================
# 4. Load Source with Column Pruning + Early Date Filter
#    OPT-1: Shipment_Date filter BEFORE window function
#    OPT-2: Select only needed columns at read time
# ========================================================
today = F.current_date()
yesterday = F.date_sub(today, 1)

# Read schema once to validate column names exist
_src = spark.table(source_table)
_available = set(_src.columns)
select_cols = [c for c in all_needed_cols if c in _available]

sni_df = (
    _src
    .select(select_cols)                                    # OPT-2: column pruning
    .filter(F.col("Shipment_Date").isin(yesterday, today))  # OPT-1: filter first
)

# ========================================================
# 5. Latest Snapshot Dedup (now runs on tiny subset)
#    OPT-3: Window operates on filtered data only instead
#    of the entire table. If Dm_Date always equals
#    current_date() for latest snapshots, this can be
#    further simplified to:
#       .filter(F.col("Dm_Date") == F.current_date())
# ========================================================
w = Window.partitionBy(
    "Sales_Order_Identifier",
    "Sales_Order_Line_Item_Identifier"
)

sni_df = (
    sni_df
    .withColumn("max_snapshot_date", F.max("Dm_Date").over(w))
    .filter(F.col("Dm_Date") == F.col("max_snapshot_date"))
    .drop("max_snapshot_date")
)

# ========================================================
# 6. CHANGE-DETECTION — Feature Fingerprint
# ========================================================
_hash_input_cols = sorted(set(
    feature_cols
    + _label_feature_cols
    + _date_cols_for_engineering
))
_hash_input_cols = [c for c in _hash_input_cols if c in sni_df.columns]

sni_df = sni_df.withColumn(
    "feature_hash",
    F.md5(F.concat_ws(
        "||",
        *[F.coalesce(F.col(c).cast("string"), F.lit("__NULL__"))
          for c in _hash_input_cols]
    )),
)

# --- compare with target table ---
if spark.catalog.tableExists(target_table):
    _existing = (
        spark.table(target_table)
        .select(*id_cols, F.col("feature_hash").alias("_prev_hash"))
    )
    _src_cols = sni_df.columns
    sni_df = (
        sni_df.alias("src")
        .join(_existing.alias("tgt"), id_cols, "left")
        .filter(
            F.col("_prev_hash").isNull()
            | (F.col("_prev_hash") != F.col("src.feature_hash"))
        )
        .select([F.col(f"src.{c}") for c in _src_cols])
    )

# ========================================================
# 6b. Early exit when nothing changed
# ========================================================
df_inference = sni_df.cache()
_rows_to_predict = df_inference.count()
print(f"\u25b6 Rows requiring prediction: {_rows_to_predict}")

if _rows_to_predict == 0:
    print("\u2705 No new or changed rows — skipping prediction & merge.")
    dbutils.notebook.exit("NO_CHANGES")

# ========================================================
# 7. Date Casting
# ========================================================
for d in _date_cols_for_engineering:
    if d in df_inference.columns:
        df_inference = df_inference.withColumn(d, F.to_timestamp(F.col(d)))

# ========================================================
# 7b. Engineer time-delta features for the label model
# ========================================================
if {"Shipment_Date", "Sales_Order_Header_Create_Date"} <= set(df_inference.columns):
    df_inference = df_inference.withColumn(
        "days_order_to_shipment",
        F.datediff(F.col("Shipment_Date"), F.col("Sales_Order_Header_Create_Date")).cast("double")
    )
if {"Proof_Of_Delivery_Date", "Shipment_Date"} <= set(df_inference.columns):
    df_inference = df_inference.withColumn(
        "days_shipment_to_pod",
        F.datediff(F.col("Proof_Of_Delivery_Date"), F.col("Shipment_Date")).cast("double")
    )
if {"Shipment_Date", "Planned_Shipment_Date"} <= set(df_inference.columns):
    df_inference = df_inference.withColumn(
        "days_planned_vs_actual_shipment",
        F.datediff(F.col("Shipment_Date"), F.col("Planned_Shipment_Date")).cast("double")
    )
if {"planned_invoice_date", "Sales_Order_Header_Create_Date"} <= set(df_inference.columns):
    df_inference = df_inference.withColumn(
        "days_order_to_planned_invoice",
        F.datediff(F.col("planned_invoice_date"), F.col("Sales_Order_Header_Create_Date")).cast("double")
    )
if {"Crdd", "Tdd"} <= set(df_inference.columns):
    df_inference = df_inference.withColumn(
        "days_crdd_minus_tdd",
        F.datediff(F.col("Crdd"), F.col("Tdd")).cast("double")
    )
if {"Actual_Quantity_Delivered", "Shipment_Quantity"} <= set(df_inference.columns):
    df_inference = (
        df_inference
        .withColumn("delivered_ratio",
            F.when(F.col("Shipment_Quantity").isNull() | (F.col("Shipment_Quantity") == 0), F.lit(0.0))
             .otherwise(F.col("Actual_Quantity_Delivered") / F.col("Shipment_Quantity"))
        )
        .withColumn("delivered_ratio",
            F.when(F.col("delivered_ratio") < 0, 0.0)
             .when(F.col("delivered_ratio") > 5, 5.0)
             .otherwise(F.col("delivered_ratio"))
        )
    )

# ========================================================
# 8. Pandas Conversion + Predictions
# ========================================================
pdf = df_inference.toPandas()

# Convert Timestamp columns in feature sets to days since 2021-01-01
# (matches datediff approach used during model training)
for c in feature_cols + _label_feature_cols:
    if c in pdf.columns and pd.api.types.is_datetime64_any_dtype(pdf[c]):
        pdf[c] = (pdf[c] - pd.Timestamp("2021-01-01")).dt.days.astype(float)

# ---------- Regression ----------
X_reg = pdf[feature_cols]
X_enc = preprocess_loaded.transform(X_reg)
pdf["Predicted_Dm_Aging_Days"] = xgb_loaded.predict(X_enc)

# ---------- Classification ----------
X_label = pdf[_label_feature_cols]
pdf["SNI_Label"] = sni_label_model.predict(X_label)

# ========================================================
# 9. Convert Back to Spark
# ========================================================
df_predicted = spark.createDataFrame(
    pdf[
        id_cols +
        ["Dm_Aging_Days", "Predicted_Dm_Aging_Days", "SNI_Label", "feature_hash"]
    ]
).dropDuplicates(id_cols)

# Add prediction_date as the current date
df_predicted = df_predicted.withColumn("prediction_date", F.current_date())

# --------------------------------------------------------
# 10. MERGE (UPSERT) into NEW Delta Table
# --------------------------------------------------------
_merge_update_cols = ["Dm_Aging_Days", "Predicted_Dm_Aging_Days", "SNI_Label", "feature_hash", "prediction_date"]
_merge_insert_cols = id_cols + _merge_update_cols

if not spark.catalog.tableExists(target_table):
    (
        df_predicted
        .write
        .format("delta")
        .saveAsTable(target_table)
    )
else:
    delta = DeltaTable.forName(spark, target_table)

    (
        delta.alias("t")
        .merge(
            df_predicted.alias("s"),
            """
            t.Sales_Order_Identifier = s.Sales_Order_Identifier AND
            t.Sales_Order_Line_Item_Identifier = s.Sales_Order_Line_Item_Identifier AND
            t.Shipment_Identifier = s.Shipment_Identifier AND
            t.Shipment_Line_Item_Identifier = s.Shipment_Line_Item_Identifier AND
            t.Shipment_Date = s.Shipment_Date
            """
        )
        .whenMatchedUpdate(set={c: f"s.{c}" for c in _merge_update_cols})
        .whenNotMatchedInsert(values={c: f"s.{c}" for c in _merge_insert_cols})
        .execute()
    )

/local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.10/site-packages/xgboost/sklearn.py:782: UserWarning: Loading a native XGBoost model with Scikit-Learn interface.
  warnings.warn("Loading a native XGBoost model with Scikit-Learn interface.")


▶ Rows requiring prediction: 4070
✅ Merged 4070 predicted rows into innovation_supplychain_stg.sni_otc.sni_prediction_regression_classification_v2


**Audit logging and run type tracking**

In [0]:
# ========================================================
# POST-PREDICTION: Audit Logging & Run Source Tracking
# Runs AFTER Cell 2 — captures run metrics, tags rows,
# and appends an audit record.  Cell 2 is NOT modified.
# ========================================================
import uuid
from datetime import datetime
import pyspark.sql.functions as F

# --------------------------------------------------------
# 1. Detect run source  (spark.databricks.job.id is only
#    set when the notebook runs as a scheduled/triggered job)
# --------------------------------------------------------
try:
    _job_id = spark.conf.get("spark.databricks.job.id")
    _run_source = "SCHEDULED_JOB"
except Exception:
    _job_id = None
    _run_source = "INTERACTIVE"

try:
    _job_run_id = spark.conf.get("spark.databricks.job.runId")
except Exception:
    _job_run_id = None

try:
    _notebook_path = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook().getContext().notebookPath().get()
    )
except Exception:
    _notebook_path = None

try:
    _cluster_id = spark.conf.get(
        "spark.databricks.clusterUsageTags.clusterId"
    )
except Exception:
    _cluster_id = None

_run_id = str(uuid.uuid4())

# --------------------------------------------------------
# 2. Add tracking columns to prediction table (one-time)
# --------------------------------------------------------
_tgt_cols = {f.name for f in spark.table(target_table).schema.fields}

if "prediction_timestamp" not in _tgt_cols:
    spark.sql(
        f"ALTER TABLE {target_table} "
        f"ADD COLUMN (prediction_timestamp TIMESTAMP "
        f"COMMENT 'Exact time of prediction run')"
    )
if "run_source" not in _tgt_cols:
    spark.sql(
        f"ALTER TABLE {target_table} "
        f"ADD COLUMN (run_source STRING "
        f"COMMENT 'SCHEDULED_JOB or INTERACTIVE')"
    )

# --------------------------------------------------------
# 3. Tag today's predicted rows with run metadata
#    Only rows predicted TODAY get the current timestamp
#    and source.  Previous days' metadata stays intact.
# --------------------------------------------------------
spark.sql(f"""
    UPDATE {target_table}
    SET prediction_timestamp = current_timestamp(),
        run_source           = '{_run_source}'
    WHERE prediction_date = current_date()
""")

print(f"\u2705 Tagged today's predictions with run_source = '{_run_source}'")

# --------------------------------------------------------
# 4. Compute source row count for audit
# --------------------------------------------------------
_source_row_count = (
    spark.table(source_table)
    .filter(
        F.col("Shipment_Date").isin(
            F.current_date(),
            F.date_sub(F.current_date(), 1),
        )
    )
    .count()
)

# --------------------------------------------------------
# 5. Ensure audit log table exists
# --------------------------------------------------------
spark.sql("""
    CREATE TABLE IF NOT EXISTS
    innovation_supplychain_stg.sni_otc.sni_prediction_run_log (
        run_id              STRING,
        run_timestamp       TIMESTAMP,
        run_source          STRING,
        job_run_id          STRING,
        job_id              STRING,
        source_row_count    LONG,
        eligible_row_count  LONG,
        changed_row_count   LONG,
        predicted_row_count LONG,
        prediction_date     DATE,
        notebook_path       STRING,
        cluster_id          STRING
    ) USING DELTA
    COMMENT 'Audit log for SNI batch prediction runs'
""")

# --------------------------------------------------------
# 6. Insert audit record
# --------------------------------------------------------
_job_run_sql = f"'{_job_run_id}'" if _job_run_id else "NULL"
_job_id_sql  = f"'{_job_id}'"     if _job_id     else "NULL"
_nb_path_sql = f"'{_notebook_path}'" if _notebook_path else "NULL"
_cl_id_sql   = f"'{_cluster_id}'"    if _cluster_id   else "NULL"

spark.sql(f"""
    INSERT INTO innovation_supplychain_stg.sni_otc.sni_prediction_run_log
    VALUES (
        '{_run_id}',
        current_timestamp(),
        '{_run_source}',
        {_job_run_sql},
        {_job_id_sql},
        {_source_row_count},
        NULL,
        {_rows_to_predict},
        {_rows_to_predict},
        current_date(),
        {_nb_path_sql},
        {_cl_id_sql}
    )
""")

print(f"\U0001f4cb Audit record inserted:")
print(f"   run_id        = {_run_id}")
print(f"   run_source    = {_run_source}")
print(f"   source_rows   = {_source_row_count:,}")
print(f"   predicted     = {_rows_to_predict:,}")
print(f"   job_run_id    = {_job_run_id}")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8876793345197449>, line 46
     41 _run_id = str(uuid.uuid4())
     43 # --------------------------------------------------------
     44 # 2. Add tracking columns to prediction table (one-time)
     45 # --------------------------------------------------------
---> 46 _tgt_cols = {f.name for f in spark.table(target_table).schema.fields}
     48 if "prediction_timestamp" not in _tgt_cols:
     49     spark.sql(
     50         f"ALTER TABLE {target_table} "
     51         f"ADD COLUMN (prediction_timestamp TIMESTAMP "
     52         f"COMMENT 'Exact time of prediction run')"
     53     )

NameError: name 'target_table' is not defined

## Monitoring & Data Validation

This section provides:
1. **Audit Log Table** — Tracks each prediction run with source counts, predicted counts, and run source (job vs interactive)
2. **Reconciliation Queries** — Compare source table vs predictions to detect discrepancies
3. **Run Analysis** — Understand when predictions happen and from which source

In [0]:
%sql
-- ========================================================
-- CREATE AUDIT LOG TABLE (run once)
-- Tracks each prediction run with metrics
-- ========================================================
CREATE TABLE IF NOT EXISTS innovation_supplychain_stg.sni_otc.sni_prediction_run_log (
  run_id STRING COMMENT 'Unique identifier for each run (UUID)',
  run_timestamp TIMESTAMP COMMENT 'Exact timestamp when prediction started',
  run_source STRING COMMENT 'SCHEDULED_JOB or INTERACTIVE',
  job_run_id STRING COMMENT 'Databricks job run ID (null if interactive)',
  job_id STRING COMMENT 'Databricks job ID (null if interactive)',
  source_row_count LONG COMMENT 'Total rows in source table for today/yesterday shipments',
  eligible_row_count LONG COMMENT 'Rows after deduplication (latest snapshot)',
  changed_row_count LONG COMMENT 'Rows that needed prediction (new or changed)',
  predicted_row_count LONG COMMENT 'Rows actually predicted and merged',
  prediction_date DATE COMMENT 'Date of prediction',
  notebook_path STRING COMMENT 'Path of notebook that ran the prediction',
  cluster_id STRING COMMENT 'Cluster ID used for the run'
)
USING DELTA
COMMENT 'Audit log for SNI batch prediction runs - tracks job vs interactive, row counts, and timing';

### Daily Prediction Summary

In [0]:
%sql
-- ========================================================
-- DAILY PREDICTION SUMMARY
-- Overview of predictions by date and run source
-- ========================================================
SELECT 
  prediction_date,
  run_source,
  COUNT(*) AS num_runs,
  SUM(predicted_row_count) AS total_rows_predicted,
  SUM(changed_row_count) AS total_changed_rows,
  MIN(run_timestamp) AS first_run,
  MAX(run_timestamp) AS last_run,
  AVG(predicted_row_count) AS avg_rows_per_run
FROM innovation_supplychain_stg.sni_otc.sni_prediction_run_log
GROUP BY prediction_date, run_source
ORDER BY prediction_date DESC, run_source;

### Source vs Prediction Reconciliation

In [0]:
%sql
-- ========================================================
-- SOURCE vs PREDICTIONS RECONCILIATION
-- Compare SNI source table row counts with prediction output
-- Helps identify missing predictions or coverage gaps
-- ========================================================
WITH source_counts AS (
  SELECT 
    CAST(Shipment_Date AS DATE) AS shipment_date,
    COUNT(*) AS source_rows,
    COUNT(DISTINCT CONCAT(
      Sales_Order_Identifier, '|',
      Sales_Order_Line_Item_Identifier, '|',
      Shipment_Identifier, '|',
      Shipment_Line_Item_Identifier
    )) AS source_unique_keys
  FROM supplychain.shipped_not_invoiced.fact_enterprise_shipped_not_invoiced_legacy_combined_reporting_view
  WHERE Shipment_Date >= CURRENT_DATE - INTERVAL 7 DAYS
  GROUP BY CAST(Shipment_Date AS DATE)
),
prediction_counts AS (
  SELECT 
    CAST(Shipment_Date AS DATE) AS shipment_date,
    COUNT(*) AS predicted_rows,
    COUNT(DISTINCT CONCAT(
      Sales_Order_Identifier, '|',
      Sales_Order_Line_Item_Identifier, '|',
      Shipment_Identifier, '|',
      Shipment_Line_Item_Identifier
    )) AS predicted_unique_keys
  FROM innovation_supplychain_stg.sni_otc.sni_prediction_regression_classification_v2
  WHERE Shipment_Date >= CURRENT_DATE - INTERVAL 7 DAYS
  GROUP BY CAST(Shipment_Date AS DATE)
)
SELECT 
  COALESCE(s.shipment_date, p.shipment_date) AS shipment_date,
  s.source_rows,
  s.source_unique_keys,
  p.predicted_rows,
  p.predicted_unique_keys,
  s.source_unique_keys - COALESCE(p.predicted_unique_keys, 0) AS missing_predictions,
  ROUND(100.0 * COALESCE(p.predicted_unique_keys, 0) / NULLIF(s.source_unique_keys, 0), 2) AS coverage_pct
FROM source_counts s
FULL OUTER JOIN prediction_counts p ON s.shipment_date = p.shipment_date
ORDER BY shipment_date DESC;

### Job vs Interactive Run analysis

In [0]:
%sql
-- ========================================================
-- JOB vs INTERACTIVE RUN ANALYSIS
-- Understand prediction patterns by run source
-- ========================================================
SELECT 
  run_source,
  COUNT(*) AS total_runs,
  SUM(predicted_row_count) AS total_predictions,
  AVG(predicted_row_count) AS avg_predictions_per_run,
  MIN(run_timestamp) AS first_run_ever,
  MAX(run_timestamp) AS most_recent_run,
  COUNT(DISTINCT prediction_date) AS distinct_days
FROM innovation_supplychain_stg.sni_otc.sni_prediction_run_log
GROUP BY run_source

UNION ALL

SELECT 
  'TOTAL' AS run_source,
  COUNT(*) AS total_runs,
  SUM(predicted_row_count) AS total_predictions,
  AVG(predicted_row_count) AS avg_predictions_per_run,
  MIN(run_timestamp) AS first_run_ever,
  MAX(run_timestamp) AS most_recent_run,
  COUNT(DISTINCT prediction_date) AS distinct_days
FROM innovation_supplychain_stg.sni_otc.sni_prediction_run_log;

###Why Source rows were not predicted

In [0]:
%sql
-- ========================================================
-- ROOT-CAUSE: Why Source Rows Are Not Predicted
-- Categorises every source row into its drop reason
-- ========================================================
WITH source_all AS (
  -- All source rows from the last 7 days with their composite key
  SELECT
    Sales_Order_Identifier,
    Sales_Order_Line_Item_Identifier,
    Shipment_Identifier,
    Shipment_Line_Item_Identifier,
    CAST(Shipment_Date AS DATE)    AS shipment_date,
    Dm_Date,
    ROW_NUMBER() OVER (
      PARTITION BY Sales_Order_Identifier,
                   Sales_Order_Line_Item_Identifier
      ORDER BY Dm_Date DESC
    ) AS rn   -- 1 = latest snapshot
  FROM supplychain.shipped_not_invoiced.fact_enterprise_shipped_not_invoiced_legacy_combined_reporting_view
  WHERE Shipment_Date >= CURRENT_DATE - INTERVAL 7 DAYS
),

prediction_keys AS (
  -- All keys already in the prediction table
  SELECT DISTINCT
    Sales_Order_Identifier,
    Sales_Order_Line_Item_Identifier,
    Shipment_Identifier,
    Shipment_Line_Item_Identifier,
    feature_hash AS existing_hash
  FROM innovation_supplychain_stg.sni_otc.sni_prediction_regression_classification_v2
),

diagnosis AS (
  SELECT
    s.*,
    p.existing_hash,
    CASE
      -- 1) Shipment date is in the future (> today)
      WHEN s.shipment_date > CURRENT_DATE
        THEN 'FUTURE_SHIPMENT_DATE'

      -- 2) Shipment date is older than yesterday → outside the
      --    today/yesterday filter in the pipeline
      WHEN s.shipment_date < CURRENT_DATE - INTERVAL 1 DAY
        THEN 'OUTSIDE_DATE_WINDOW'

      -- 3) Not the latest snapshot → dropped by dedup step
      WHEN s.rn > 1
        THEN 'SUPERSEDED_SNAPSHOT'

      -- 4) Already predicted with same features → skipped by
      --    change-detection (feature_hash match)
      WHEN p.existing_hash IS NOT NULL
        THEN 'UNCHANGED_FEATURES'

      -- 5) Should have been predicted — investigate further
      WHEN p.existing_hash IS NULL AND s.rn = 1
           AND s.shipment_date BETWEEN CURRENT_DATE - INTERVAL 1 DAY
                                   AND CURRENT_DATE
        THEN 'GENUINELY_MISSING'

      ELSE 'OTHER'
    END AS drop_reason
  FROM source_all s
  LEFT JOIN prediction_keys p
    ON  s.Sales_Order_Identifier          = p.Sales_Order_Identifier
    AND s.Sales_Order_Line_Item_Identifier = p.Sales_Order_Line_Item_Identifier
    AND s.Shipment_Identifier             = p.Shipment_Identifier
    AND s.Shipment_Line_Item_Identifier   = p.Shipment_Line_Item_Identifier
)

SELECT
  shipment_date,
  drop_reason,
  COUNT(*) AS row_count
FROM diagnosis
GROUP BY shipment_date, drop_reason
ORDER BY shipment_date DESC, row_count DESC;